# E1 Distillation Baseline
Author: Arush Arora

## Introduction
This notebook explores the development of a GREP-PRISM Classical Planning LLM. The workflow for the GREP-PRISM model includes the following steps:
1. Tokenize a classical planning prompt that includes a relevant scene graph in text.
2. Obtain embeddings from a trained R-PEARL model that produces Graph Positional Encodings (GREPs) in $\mathbb{R}^d$ and add them to select word embeddings from the prompt semantically representing nodes in the scene graph.
3. Feed the prompt, containing a mix of Fourier and graphically positioned word embeddings, to the distilled Llama3.2:0.5b PRISM model that will process the Classical Planning prompt without the scene graph to return the next action of the robot.

_Note_: The training loop will remove the final softmax layer of the transformer for Cross-Entropy Loss evaluation.

## Libraries

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import gc
import torch

def clear_gpu_memory():
    # 1. Delete the model/trainer variables if they exist in global scope
    # globals() checks ensure we don't error if the variable isn't defined
    if 'model' in globals(): del globals()['model']
    if 'trainer' in globals(): del globals()['trainer']

    # 2. Run Garbage Collection to release Python references
    gc.collect()

    # 3. Clear PyTorch's CUDA cache
    torch.mps.empty_cache()

    # 4. Optional: Verify
    # print(f"GPU Memory Allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")

# Run this before your training block
clear_gpu_memory()

In [3]:
import json
from ast import literal_eval
import string
import re

In [4]:
import gc
import torch

device = 'mps'

def find_cuda_tensors():
    tensors = []
    for obj in gc.get_objects():
        try:
            if torch.is_tensor(obj) or (hasattr(obj, 'data') and torch.is_tensor(obj.data)):
                # Check if the tensor is on a CUDA device
                if obj.is_mps:
                    tensors.append(obj)
        except Exception:
            pass # ignore errors from objects that are not tensors or do not have is_cuda attribute

    return tensors

## The R-PEARL GNN

The Random Positional Encoding (R-PEARL) GNN architecture is a PE generator that inputs white noise and processes it over an undirected graph $\mathcal{G} = (\mathcal{V}, \mathcal{E}, \mathcal{W})$. In this work, the graph is represented by an adjacency matrix $\utilde{A}$, and the GNN composes [Topology Adaptive Graph (TAG)](https://arxiv.org/abs/1710.10370) Convolutional Layers with pointwise nonlinearities (demodulators).

### Graph Convolutional Network (GNN)
The code below establishes this project's implementation of a Graph Convolutional Network, which is the foundational architecture comprising R-PEARL. The equation to demonstrate the internal architecture of this NN as follows (in most cases, $P(\cdot) = I(\cdot)$, where $I$ is the identity function):
$\renewcommand{\utilde}[1]{\underset{\sim}{#1}}$
$$\Phi(\utilde{X}, \utilde{S}, \mathcal{H}) = \utilde{X}^{(L)}$$
$$\utilde{X}^{(0)} = \utilde{X} \qquad \utilde{X}^{(l)} = P\Bigg[\sigma\Bigg(\sum_{k = 0}^{K^{(l)} - 1} \utilde{S}^k\utilde{X}^{(l - 1)}{\utilde{H}}_k^{(l)}\Bigg)\Bigg]$$

In [5]:
import torch

from torch import nn
from torch.utils.checkpoint import checkpoint
from torch_geometric.nn import TAGConv
from torch_geometric.data import Data

We add a device for management-purposes.

In [6]:
class GCN(nn.Module):
    """
    A simple TAG-based graph convolutional backbone that returns node embeddings.

    Args:
        in_channels (int): Number of input features per node
        hidden_channels (int): Number of hidden features per node
        num_layers (int): Number of convolution layers (must be >= 2)
        skip_connection (bool): Whether to use skip connections
        use_batch_norm (bool): Whether to use batch normalization
        k (int): Order of TAGConv polynomial (K)

    Returns:
        torch.Tensor: Node embeddings of shape [num_nodes, hidden_channels]
    """

    def __init__(self,
        in_channels,
        hidden_channels,
        num_layers,
        skip_connection=False,
        use_batch_norm=False,
        dropout=0.5,
        k: int = 3,
    ):
        super().__init__()
        if num_layers < 2:
            raise ValueError("GCN requires at least 2 layers.")

        self.convs = nn.ModuleList()
        self.k = k
        self.convs.append(TAGConv(in_channels, hidden_channels, K=self.k))
        self.norms = nn.ModuleList()
        for _ in range(num_layers - 2):
            self.convs.append(TAGConv(hidden_channels, hidden_channels, K=self.k))
            self.norms.append(nn.LayerNorm(hidden_channels))
            if use_batch_norm:
                self.norms.append(nn.BatchNorm1d(hidden_channels))
        self.convs.append(TAGConv(hidden_channels, hidden_channels, K=self.k))
        self.relu = nn.LeakyReLU()
        self.dropout = nn.Dropout(p=dropout)
        self.skip_connection = skip_connection
        self.embedding_dim = hidden_channels

    def forward(self, data: Data):
        """
        Forward pass through the GCN.

        Args:
            data (Data): PyTorch Geometric Data object containing node features (x)
                        and edge indices (edge_index)

        Returns:
            torch.Tensor: Output node embeddings [num_nodes, hidden_channels]
        """
        device = next(self.parameters()).device
        data.x = data.x.to(device)
        data.edge_index = data.edge_index.to(device)
        x0, edge_index = data.x, data.edge_index
        x_prev = x0
        x = x0.clone()
        for i, conv in enumerate(self.convs[:-1]):
            x = conv(x_prev, edge_index)
            if i < len(self.norms):
                x = self.norms[i](x)
            x = self.relu(x)
            x = self.dropout(x)
            if self.skip_connection and i > 0:
                x = x + x_prev
            x_prev = x
        x = self.convs[-1](x, edge_index)
        return x

### Random Graph Positional Encodings (R-PEARL)
The R-PEARL architecture extends on the GCN by instantiating it with simply one layer – a TAG Convolution and Demodulator. The mathematical equations below express the functionality of the R-PEARL network:
1. The white-noise matrix is sampled from the Gaussian distribution. $\renewcommand{\utilde}[1]{\underset{\sim}{#1}}$ $$\utilde{Q} \in \mathbb{R}^{M \times N} \qquad \utilde{Q} \sim \mathcal{N}(0, \utilde{I}) \qquad \utilde{Q} = \begin{bmatrix}
  \mathbf{q}^{(0)} & \cdots & \mathbf{q}^{(m)} & \cdots & \mathbf{q}^{(M)}
  \end{bmatrix}$$

2. The R-PEARL network has row-vector parameter $\utilde{H}^{(0)} \in \mathbb{R}^{1 \times D}$. It takes in each column of the white-noise matrix individually and produces a sample $\utilde{P}^{(m)} \in \mathbb{R}^{N \times D}$, which are then pooled to form GREP $\utilde{P}$:$$\utilde{P}^{(m)} = \Phi\Big(\mathbf{q}^{(m)}, \utilde{S}, \mathcal{H}\Big) = \sigma\bigg(\sum_{k = 0}^{K = 1} \utilde{S}^k\mathbf{q}^{(m)} {\utilde{H}}_k\bigg)$$$$\utilde{P} = \hat{\mathbb{E}}\Big[\mathbf{p}^{(m)}\Big] = \frac{1}{M}\sum_{m = 1}^{M} \utilde{P}^{(m)}$$

In [7]:
class RandomGNNPositionalEncodings(nn.Module):
    """
    Random graph positional encodings (R-PEARL).

    Args:
        pe_hidden_channels (int): Hidden dimension for the GCN
        pe_num_layers (int): Number of layers in the GCN
        d_model (int): Output dimension
        num_samples (int): Number of random samples (M) to use
        use_layer_norm (bool): Whether to use layer normalization
    """

    def __init__(self,
        pe_hidden_channels,
        pe_num_layers,
        d_model,
        num_samples=30,
        dropout=0.1,
        use_layer_norm=False,
    ):
        super().__init__()
        # Create a GCN that takes 1-dimensional random features
        self.pe_gcn = GCN(
            1, pe_hidden_channels, pe_num_layers, skip_connection=True, dropout=dropout
        )
        # Add a final projection to ensure output is d_model dimensions
        self.output_projection = nn.Linear(pe_hidden_channels, d_model)
        self.dropout = nn.Dropout(dropout)
        self.use_layer_norm = use_layer_norm
        if self.use_layer_norm:
            self.layer_norm = nn.LayerNorm(d_model)
        else:
            self.batch_norm = nn.BatchNorm1d(d_model)
        self.M = num_samples

    def forward(self, data):
        # Move input data to the model's device.
        device = next(self.parameters()).device
        data.x = data.x.to(device)
        data.edge_index = data.edge_index.to(device)
        X, edge_index = data.x, data.edge_index

        # Generate random node embeddings for positional encoding
        num_nodes = X.shape[0]
        Q = torch.randn((num_nodes, self.M), device=device)

        # Process random embeddings individually through GCN
        P_m = []

        for i in range(self.M):

            def _pe_block(q_col, edge_idx, _dummy):
                q_data = Data(x=q_col.unsqueeze(-1), edge_index=edge_idx)
                pe_local = self.pe_gcn(q_data)
                pe_local = self.dropout(pe_local)
                pe_local = self.output_projection(pe_local)
                return pe_local

            dummy = Q.new_ones(1, requires_grad=True, device=device)
            pe = checkpoint(_pe_block, Q[:, i], edge_index, dummy, use_reentrant=False)
            P_m.append(pe)
        # checkpoint

        P = torch.stack(P_m, dim=-1)
        pooled_pe = P.mean(dim=-1)
        if self.use_layer_norm:
            pooled_pe = self.layer_norm(pooled_pe)
        else:
            pooled_pe = self.batch_norm(pooled_pe)
        return pooled_pe

## Transformer

$\renewcommand{\utilde}[1]{\underset{\sim}{#1}}$The Transformer architecture follows that of the Llama3.2-3B distilled PRISM model. First, the TXT file, containing the scene-graph data, is tokenized and embedded into matrices $\utilde{E}$ and $\utilde{\tilde{X}}$ as follows, where $V$ is the size of the vocabulary and $d$ is the embedding dimension.

$$\text{TXT Tokenized Data from GPT-4: } \utilde{E} = \begin{bmatrix}
\mathbf{e}_1 & \mathbf{e}_2 & \overset{\mathbf{e}_t}{\cdots} & \mathbf{e}_T
\end{bmatrix}^\top \qquad \mathbf{e}_t \in \mathbb{R}^V$$

$$\text{Embed: } \utilde{X} = \begin{bmatrix}
\mathbf{x}_1 & \mathbf{x}_2 & \overset{\mathbf{x}_t}{\cdots} & \mathbf{x}_T
\end{bmatrix}^\top \qquad \mathbf{x}_t \in \mathbb{R}^d$$

Next, the transformer operates using the equations below:

$$\utilde{X} = \utilde{\tilde{X}} + \utilde{P}$$

$${\utilde{Z}}_{1:t}^{(L)} = \operatorname{Trf}\bigg({\utilde{X}}_{1:t}, {\mathcal{T}}_l\bigg) \qquad {\mathcal{T}}_l = \begin{bmatrix}
{\utilde{Q}}_l & {\utilde{K}}_l & {\utilde{V}}_l & \left({\utilde{W}}_o\right)_l
\end{bmatrix}^\top \in \mathbb{R}^{4 \times T \times D}$$

$$\hat{\mathbf{Y}}_{t + 1} = \operatorname{Linear}\Big({\utilde{Z}}_{1:t}^{(L)}\Big) \in \mathbb{R}^V$$
$$\text{Cross-Entropy Loss: } \mathcal{L}(\utilde{E}, \hat{\mathbf{Y}}) = \sum_t \sum_v e_{vt}\log{\hat{y}_t}$$

In [8]:
from huggingface_hub import whoami

whoami()

{'type': 'user',
 'id': '6935e7c3462d178c8443d18c',
 'name': 'arar1234',
 'fullname': 'A A',
 'isPro': False,
 'avatarUrl': '/avatars/d6b9b5a20a47363d566920adb7b9338f.svg',
 'orgs': [],
 'auth': {'type': 'access_token',
  'accessToken': {'displayName': 'Personal',
   'role': 'fineGrained',
   'createdAt': '2026-01-06T01:47:39.606Z',
   'fineGrained': {'canReadGatedRepos': True,
    'global': [],
    'scoped': [{'entity': {'_id': '6935e7c3462d178c8443d18c',
       'type': 'user',
       'name': 'arar1234'},
      'permissions': ['inference.serverless.write']}]}}}}

In [9]:
from transformers import AutoModelForCausalLM, AutoTokenizer

models = {
    'qwen': ('Qwen/Qwen2.5-0.5B-Instruct', 896),
    'llama': ('meta-llama/Llama-3.2-3B-Instruct', 3072)
}

code = 'qwen'

name, emb_dim = models[code]
model = AutoModelForCausalLM.from_pretrained(name)
tokenizer = AutoTokenizer.from_pretrained(name)
tokenizer.pad_token = tokenizer.eos_token

model, tokenizer

(Qwen2ForCausalLM(
   (model): Qwen2Model(
     (embed_tokens): Embedding(151936, 896)
     (layers): ModuleList(
       (0-23): 24 x Qwen2DecoderLayer(
         (self_attn): Qwen2Attention(
           (q_proj): Linear(in_features=896, out_features=896, bias=True)
           (k_proj): Linear(in_features=896, out_features=128, bias=True)
           (v_proj): Linear(in_features=896, out_features=128, bias=True)
           (o_proj): Linear(in_features=896, out_features=896, bias=False)
         )
         (mlp): Qwen2MLP(
           (gate_proj): Linear(in_features=896, out_features=4864, bias=False)
           (up_proj): Linear(in_features=896, out_features=4864, bias=False)
           (down_proj): Linear(in_features=4864, out_features=896, bias=False)
           (act_fn): SiLUActivation()
         )
         (input_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
         (post_attention_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
       )
     )
     (norm): Qwen2RMSNorm((896,), eps=1e-06)
 

From `modular_qwen2.py`'s forward pass:
```python
@deprecate_kwarg("past_key_value", new_name="past_key_values", version="4.58")
def forward(
    self,
    hidden_states: torch.Tensor,
    position_embeddings: tuple[torch.Tensor, torch.Tensor],
    attention_mask: Optional[torch.Tensor],
    past_key_values: Optional[Cache] = None,
    cache_position: Optional[torch.LongTensor] = None,
    **kwargs: Unpack[FlashAttentionKwargs],
) -> tuple[torch.Tensor, Optional[torch.Tensor]]:
    ...
```

## Graph-Augmented LLM

The last class that is needed to create the full GREP-PRISM architecture is the `GraphAugmentedLLM`, which simply implements the following equation as a Neural Network object in PyTorch's `torch.nn` module (referring to above equations for definitions).
$\renewcommand{\utilde}[1]{\underset{\sim}{#1}}$ $$\utilde{P} = \hat{\mathbb{E}}\Big[\mathbf{p}^{(m)}\Big] = \frac{1}{M}\sum_{m = 1}^{M} \Phi\Big(\mathbf{q}^{(m)}, \utilde{S}, \mathcal{H}\Big)$$

$$\utilde{X} = \utilde{\tilde{X}} + \utilde{P}$$

$${\utilde{Z}}_{1:t}^{(L)} = \operatorname{Trf}\bigg({\utilde{X}}_{1:t}\bigg)$$

In [10]:
from collections import defaultdict


class GraphAugmentedLLM(nn.Module):
    """
    Graph-Augmented LLM (GREP-PRISM).

    Args:
        llm (nn.Module): LLM to perform classical planning.
        pe_model (nn.Module): R-PEARL positional-encodings model.
        tokenizer: (nn.Module): Tokenizer associated with LLM.
    """

    def __init__(self, llm: nn.Module, pe_model: nn.Module, tokenizer: nn.Module):
        super().__init__()
        self.llm = llm
        self.pe_model = pe_model
        self.config = llm.config
        self.tokenizer = tokenizer

    def __getattr__(self, name):
        try:
            return super().__getattr__(name)  # defer to nn.Module first
        except AttributeError:
            return getattr(self.llm, name)

    def forward(
        self,
        input_ids: torch.Tensor | None = None,
        attention_mask: torch.Tensor | None = None,
        labels: torch.Tensor | None = None,
        graphs: list | None = None,
        **kwargs,
    ):
        # Associate full words to token indices.
        bucket = self.bucketize_prompt(input_ids, self.tokenizer)

        # Get positional encodings.
        graph = graphs[0]
        pe = self.pe_model(graph)
        pos_enc = defaultdict(lambda: torch.Tensor(size=pe.size(), device=pe.device))
        for i, word in enumerate(graph.node_names):
            pos_enc[word] = pe[i]

        # Add positional encodings to embeddings.
        embeddings = (
            self.llm.get_input_embeddings()(input_ids)
                .squeeze(0)
                .to(pe.device)
        )
        for word, token in bucket.items():
            if word in pos_enc:
                for pos in token:
                    embeddings[pos] += pos_enc[word]

        return self.llm(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels,
            **kwargs,
        )

    @classmethod
    def bucketize_prompt(cls, input_ids: torch.Tensor | None, tokenizer: nn.Module) -> defaultdict:
        """
        Helper function for associating full prompt words with their corresponding token indices.
        Uses parallel iteration through words alongside the token list.

        Args:
            input_ids (torch.Tensor): List of one-hot encodings for prompt.
            tokenizer (nn.Module): LLM tokenizer required to decode input IDs.

        Returns:
            bucket (dict): mappings for adding operation of positional encodings
                to respective tokens.
        """

        # Get prompt and token list.
        words = re.findall(rf'\b[\w_]+\b',
                            tokenizer.decode(input_ids.squeeze()))
        tokens = tokenizer.convert_ids_to_tokens(input_ids.squeeze())

        # Get map of words to token locations.
        bucket = defaultdict(list)
        j = 0
        for word in words:
            while not (cls.has_prefix_suffix_match(word, tokens[j]) or tokens[j] in word or word in tokens[j]):
                j += 1
            if cls.has_prefix_suffix_match(word, tokens[j]):
                bucket[word].append(j)
                j += 1
                while tokens[j] in bucket or cls.has_prefix_suffix_match(tokens[j], word):
                    bucket[word].append(j)
                    j += 1
            elif tokens[j] in word or word in tokens[j]:
                bucket[word].append(j)
                j += 1
        return bucket

    @staticmethod
    def has_prefix_suffix_match(a: str, b: str) -> bool:
        """Returns True if any prefix of a matches any suffix of b."""
        # Check all possible prefixes of a against suffixes of b
        for i in range(1, len(a)+1):
            prefix_a = a[:i]
            for j in range(1, len(b)+1):
                suffix_b = b[-j:]
                if prefix_a == suffix_b:
                    return True
        return False

## Data Processing

After defining the internal mechanisms of the GREP-PRISM architecture employed in this project, we turn to the processing of all Classical Planning prompts in `gpt_gen_formatted.json` using the scene-graph decoding algorithms included in this work's parent project, PRISM.

### Data Loading

We now have all architectures needed to run the full training algorithm. We begin by importing all Classical Planning prompt data with the scene graphs attached.

In [11]:
with open("data/eval/gpt_gen_formatted.json", "r") as f:
    raw = json.load(f)
# Take the first element of the first conversation to check out that scene graph.
first_prompt = raw[0]['conversations'][0]['content']
display(first_prompt[:100])

scene_graph_text = re.findall(r"Scene graph:(.*)", first_prompt)[0]
display(scene_graph_text[:100])

graph_dict = literal_eval(scene_graph_text)  # handles the single quotes safely
display(graph_dict.keys())

"task: I need a shovel. Is there one in the scene?Scene graph:{'objects': [{'name': 'house_1', 'coord"

"{'objects': [{'name': 'house_1', 'coords': [-1, -1]}, {'name': 'house_2', 'coords': [-3, -1]}, {'nam"

dict_keys(['objects', 'regions', 'object_connections', 'region_connections', 'robot_location'])

In [12]:
from trl.trainer import SFTTrainer, SFTConfig
from datasets import load_dataset
from prism import scene_graph_parser

train_dataset = load_dataset("json", data_files=["data/eval/gpt_gen_formatted.json"], split="train")


def _add_messages(example):
    example["messages"] = example["conversations"]
    return example


def _tokenize_with_conversations(example):
    tokenized = tokenizer.apply_chat_template(
        example["messages"], tokenize=True, return_dict=True
    )
    tokenized["conversations"] = example["conversations"]
    tokenized["messages"] = example["messages"]
    return tokenized


train_dataset = train_dataset.map(_add_messages)
train_dataset = train_dataset.map(_tokenize_with_conversations)
#  train_dataset = train_dataset.map(scene_graph_parser._parse_scene_graph_dictionary_from_conversation)

train_dataset

Dataset({
    features: ['conversations', 'messages', 'input_ids', 'attention_mask'],
    num_rows: 990
})

### Accomodating Erroneous Graphs

We see that SPINE's `parse_graph` function may not be enough to process eroneous graphs. We thus define a function that converts a JSON graph into an `nx.Graph` object from the package `networkx`. We configure the `safe_parse_graph` function to handle graphs that may be incorrectly defined (missing nodes in the edge list, etc.).

In [13]:
from typing import Dict, Optional, Tuple
from scipy.spatial.transform import Rotation
import networkx as nx
import numpy as np
from copy import deepcopy
from spine.mapping.graph_util import parse_graph_coord

def safe_parse_graph(
    data: Dict[str, Dict[str, str]],
    custom_data: Optional[Dict[str, Dict[str, str]]] = {},
    rotation: Optional[Rotation] = None,
    utm_origin: Optional[np.ndarray] = None,
    flip_coords=False,
) -> Tuple[nx.Graph, str]:
    """Parse scene graph in `data` into a networkx object.

    Parameters
    ----------
    data : Dict[str, Dict[str, str]]
        graph where keys-values are nodes-attributes
    rotation : Optional[Rotation]
        current rotation of robot

    Returns
    -------
    Tuple[nx.Graph, str]
        Networkx and string of json
    """
    origin = np.array([0, 0])
    data = deepcopy(data)  # don't modify input data
    as_str = str(data)

    if utm_origin is not None:
        origin = utm_origin

    if len(custom_data):
        add_keys = ["regions", "region_connections", "objects", "object_connections"]
        for key in add_keys:
            if key in data and key in custom_data:
                data[key].extend(custom_data[key])

    G = nx.Graph()
    for node in data["objects"]:
        coords = parse_graph_coord(node["coords"], origin=origin, rotation=rotation)
        if flip_coords:
            raise ValueError()
            # print("flipping coords")
            coords = [coords[0], -coords[1]]

        node.pop("coords")
        name = node.pop("name")
        G.add_node(name, coords=coords, type="object", **node)

    for node in data["regions"]:
        assert "coords" in node, node
        c = node["coords"]
        # print(f"node: {node}, coords: {c}")
        coords = parse_graph_coord(node["coords"], origin=origin, rotation=rotation)

        if flip_coords:
            raise ValueError
            # print("flipping coords")
            coords = [coords[0], -coords[1]]
        node.pop("coords")
        name = node.pop("name")
        G.add_node(name, coords=coords, type="object", **node)

    for edge in data["object_connections"]:
        c1 = G.nodes[edge[0]]["coords"]
        c2 = G.nodes[edge[1]]["coords"]
        # print(f"edge: {edge}, c1, c2: {c1}, {c2}")
        dist = np.linalg.norm(np.array(c1) - np.array(c2))
        G.add_edge(edge[0], edge[1], type="object", weight=dist)

    for edge in data["region_connections"]:
        c1 = G.nodes[edge[0]]["coords"]
        c2 = G.nodes[edge[1]]["coords"]
        # print(f"edge: {edge}, c1, c2: {c1}, {c2}")
        dist = np.linalg.norm(np.array(c1) - np.array(c2))
        G.add_edge(edge[0], edge[1], type="region", weight=dist)

    return G, as_str

### Creating the Data Collator

We now engineer the penultimate class to begin the training sequence. The `DataCollatorForGraphAgumentedLLM` implementation below filters scene graphs from text-based Classical Planning prompts, safely parses them into `nx.Graph` objects using the `networkx` library in Python, and sanitizes them to be fit to the Task Planning scenario. It then batches the data in preparation for GREP-PRISM training.

In [14]:
from transformers.data.data_collator import DataCollatorForLanguageModeling
import torch
import torch_geometric.utils as pyg_utils


class DataCollatorForGraphAugmentedLLM(DataCollatorForLanguageModeling):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)

    def __call__(self, features, return_tensors: Optional[str] = None):
        """Attach parsed PyG graphs for each conversation example."""
        messages = []
        pyg_graphs = []
        conversations = []
        sanitized_examples = []

        for example in features:
            pattern = r"[Ss]cene graph:"
            # Graph is in the first message
            prompt = self.tokenizer.decode(example['input_ids'])
            if re.search(pattern=pattern, string=prompt):
                scene_graph_text = re.findall(pattern + r" ?(.*})", prompt)[0]
                scene_graph_dict = literal_eval(scene_graph_text)  # handles the single quotes safely
            else:
                raise ValueError(f"No scene graph found in prompt: {prompt}")

            try:
                nx_graph, _ = safe_parse_graph(scene_graph_dict)
                node_names = list(nx_graph.nodes)
                coords = torch.tensor(
                    [nx_graph.nodes[node]["coords"] for node in node_names],
                    dtype=torch.float32,
                )

                pyg_graph = pyg_utils.from_networkx(nx_graph)
                pyg_graph.coords = coords
                pyg_graph.x = torch.zeros((coords.size(0), 1), dtype=torch.float32)
                pyg_graph.edge_index = pyg_graph.edge_index
                pyg_graph.node_names = node_names
                pyg_graph.node_types = [nx_graph.nodes[node]["type"] for node in node_names]
                pyg_graph.robot_location = scene_graph_dict.get("robot_location")
                pyg_graph.raw_scene_graph = scene_graph_dict
                pyg_graphs.append(pyg_graph)

                # Sanitize input IDs and attention masks.
                pattern = r"'object_connections':"
                decoded = self.tokenizer.decode(example["input_ids"])
                cleaned = re.sub(pattern + r' ?.*,', '', decoded)
                encoded = self.tokenizer(cleaned, return_tensors="pt")
                example['input_ids'] = encoded['input_ids'].squeeze().tolist()
                example['attention_mask'] = encoded['attention_mask'].squeeze().tolist()

                # Sanitize conversations and messages for later reinstallation.
                """
                if 'conversations' in example.keys() and messages in example.keys():
                    conv, mes = example['conversations'], example['messages']
                    conv[0]['content'] = re.sub(pattern + r' ?.*,', '', conv[0]['content'])
                    mes[0]['content'] = re.sub(pattern + r' ?.*,', '', mes[0]['content'])
                    conversations.append(conv)
                    messages.append(mes)
                """

                sanitized_examples.append(
                    {
                        k: v
                        for k, v in example.items()
                        if k not in {"conversations", "scene_graph", "messages"}
                    }
                )
            except Exception as e:
                print(f"Error parsing scene graph: {e}")
        # Call the parent collator to get the tensors (on sanitized examples so that it doesn't try to tensorize the scene graph/text)
        batch = super().__call__(sanitized_examples)
        batch["graphs"] = pyg_graphs
        if conversations and messages:
            batch['conversations'] = conversations
            batch['messages'] = messages
        return batch

In [15]:
collator = DataCollatorForGraphAugmentedLLM(tokenizer=tokenizer, mlm=False)
data = collator([train_dataset[i] for i in range(4)])
data

Error parsing scene graph: 'shed_2'


{'input_ids': tensor([[151644,   8948,    198,  ...,     92, 151645,    198],
        [151644,   8948,    198,  ..., 151645, 151645, 151645],
        [151644,   8948,    198,  ..., 151645, 151645, 151645]]), 'attention_mask': tensor([[1, 1, 1,  ..., 1, 1, 1],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0]]), 'labels': tensor([[151644,   8948,    198,  ...,     92,   -100,    198],
        [151644,   8948,    198,  ...,   -100,   -100,   -100],
        [151644,   8948,    198,  ...,   -100,   -100,   -100]]), 'graphs': [Data(
  edge_index=[2, 18],
  coords=[12, 2],
  type=[12],
  edge_type=[18],
  weight=[18],
  num_nodes=12,
  x=[12, 1],
  node_names=[12],
  node_types=[12],
  robot_location='example_road_1',
  raw_scene_graph={
    objects=[6],
    regions=[6],
    object_connections=[6],
    region_connections=[3],
    robot_location='example_road_1',
  }
), Data(
  edge_index=[2, 8],
  coords=[7, 2],
  type=[7],
  edge_type=[8],
  weight=[8],
  num_nodes=7,
  x=[

In [16]:
"""Testing"""

pe_model = RandomGNNPositionalEncodings(
    pe_hidden_channels=256, pe_num_layers=3, d_model=emb_dim, num_samples=40, dropout=0.1
)

graph_augmented_model = GraphAugmentedLLM(model, pe_model, tokenizer)

prompt = ("<|im_start|>system\nYou are a robotic agent who must complete tasks in a world.<|im_end|>\n"
          "<|im_start|>user\nGo to office_building_1 and using example_truck_1. Explicitly commend on "
          "the plan you will use to complete this task, and list your steps in bulletized fashion.<|im_end|>\n"
          "<|im_start|>assistant\n")

graph = data['graphs'][0]
graph

Data(
  edge_index=[2, 18],
  coords=[12, 2],
  type=[12],
  edge_type=[18],
  weight=[18],
  num_nodes=12,
  x=[12, 1],
  node_names=[12],
  node_types=[12],
  robot_location='example_road_1',
  raw_scene_graph={
    objects=[6],
    regions=[6],
    object_connections=[6],
    region_connections=[3],
    robot_location='example_road_1',
  }
)

In [17]:
pe_model = pe_model.to(device)
pe_model(graph)

tensor([[-2.4090e-01, -5.5498e-01, -4.0254e-01,  ..., -2.2561e-01,
         -1.6537e-01, -4.7313e-02],
        [ 7.1076e-04, -6.9878e-01, -5.0402e-01,  ..., -3.3055e-03,
         -1.9909e-01, -1.2880e-01],
        [ 8.0670e-01,  1.3299e-01,  6.0237e-02,  ..., -6.7019e-01,
          1.5134e-02,  2.9841e-02],
        ...,
        [ 1.5535e+00,  1.7298e+00, -1.0019e+00,  ...,  1.1051e+00,
          1.0479e+00, -5.9730e-01],
        [-1.5316e+00, -1.0278e+00,  9.7529e-01,  ..., -1.7201e+00,
         -1.6995e+00,  2.0304e+00],
        [-1.4146e+00, -9.4189e-01,  2.7204e+00,  ..., -1.0477e+00,
         -1.5728e+00,  1.7861e+00]], device='mps:0',
       grad_fn=<NativeBatchNormBackward0>)

In [18]:
emb = model.get_input_embeddings()
tokenized = tokenizer(prompt, return_tensors='pt')
tokenized.input_ids = tokenized.input_ids
embeddings = emb(tokenized.input_ids)
embeddings = embeddings.squeeze().to(device)
embeddings.shape

torch.Size([63, 896])

In [19]:
embeddings

tensor([[ 0.0014, -0.0084,  0.0073,  ..., -0.0032, -0.0128,  0.0187],
        [ 0.0293,  0.0210,  0.0073,  ..., -0.0014, -0.0273, -0.0216],
        [-0.0077,  0.0312, -0.0193,  ...,  0.0177,  0.0029, -0.0125],
        ...,
        [ 0.0014, -0.0084,  0.0073,  ..., -0.0032, -0.0128,  0.0187],
        [ 0.0037,  0.0215,  0.0232,  ..., -0.0165,  0.0001, -0.0109],
        [-0.0077,  0.0312, -0.0193,  ...,  0.0177,  0.0029, -0.0125]],
       device='mps:0', grad_fn=<ToCopyBackward0>)

In [20]:
bucket = graph_augmented_model.bucketize_prompt(tokenized.input_ids, tokenizer)
bucket

defaultdict(list,
            {'im_start': [0, 18, 60],
             'system': [1],
             'You': [3],
             'are': [4],
             'a': [5, 13],
             'robotic': [6],
             'agent': [7],
             'who': [8],
             'must': [9],
             'complete': [10, 45],
             'tasks': [11],
             'in': [12, 53],
             'world': [14],
             'im_end': [16, 58],
             'user': [19],
             'Go': [21],
             'to': [22, 44],
             'office_building_1': [23],
             'and': [27, 49],
             'using': [28],
             'example_truck_1': [29],
             'Explicitly': [35, 36],
             'commend': [37],
             'on': [38],
             'the': [39],
             'plan': [40],
             'you': [41],
             'will': [42],
             'use': [43],
             'this': [46],
             'task': [47],
             'list': [50],
             'your': [51],
             'steps': [52],
  

In [21]:
pe = graph_augmented_model.pe_model(graph)
pos_enc = defaultdict(lambda: torch.Tensor(size=pe.size(), device=pe.device))
for i, word in enumerate(graph.node_names):
    pos_enc[word] = pe[i]
pos_enc

defaultdict(<function __main__.<lambda>()>,
            {'office_building_1': tensor([ 5.2063e-02, -7.3212e-01, -1.1761e-01,  3.0529e-02, -8.8431e-01,
                     -3.7453e-01, -2.3412e-02, -9.5110e-01,  1.4345e+00,  1.4316e+00,
                     -2.5172e-01,  1.9672e-01,  4.2720e-01,  2.4636e-01,  3.6499e-01,
                     -1.0908e+00, -6.1135e-01, -1.6978e-01, -3.4034e-01, -8.2353e-01,
                      8.4803e-02,  9.6122e-01, -1.0189e-01, -3.5849e-02, -3.6169e-01,
                     -1.3861e-01,  6.9407e-01, -3.3831e-01, -1.2272e+00,  3.6802e-01,
                     -1.0609e-01,  3.6975e-02, -2.8994e-01,  3.8017e-01, -9.7970e-01,
                     -4.5867e-01,  8.7076e-01,  1.5029e-01,  4.6435e-02,  4.6960e-01,
                     -3.0037e-01, -1.1962e+00, -2.0170e-01, -3.1641e-01, -1.1059e-01,
                     -5.9522e-01, -1.9801e-01, -7.4507e-02,  3.3286e-01,  2.2851e-01,
                      1.3734e-01, -1.7506e-01,  7.7281e-03, -1.5137e-01, -2

In [22]:
new_embeddings = embeddings.clone()

for word, token in bucket.items():
    if word in pos_enc:
        for pos in token:
            new_embeddings[pos] += pos_enc[word]

(new_embeddings * embeddings).sum()

tensor(13.1659, device='mps:0', grad_fn=<SumBackward0>)

In [23]:
output = graph_augmented_model(
    input_ids=tokenized.input_ids,
    num_return_sequences=1,
    attention_mask=tokenized.attention_mask,
    pad_token_id=tokenizer.eos_token_id,
    graphs=[graph]
)

# output_text = output
output_text = torch.argmax(output['logits'], dim=-1)

tokenizer.decode(output_text.squeeze(), skip_special_tokens=True)

'/API\n# are given helpful assistant. has respond a for a given where YousystemIn\n\nI to the hours and1 and open the_ay_1,ly state the the efficiency and took take to get the task. and the the steps. a format format.\nuser\nI'

In [24]:
prompt = """
System: you are a robotic agent who must complete tasks in a world.
User: Go to office_building_1 and using example_truck_1. Explicitly commend on the plan you will use to complete this task, and list your steps in bulletized fashion.
Assistant: """

prompt_ids = tokenizer(prompt, return_tensors="pt").input_ids
ids = torch.tensor(train_dataset['input_ids'][0], dtype=torch.int)
bucket = GraphAugmentedLLM.bucketize_prompt(prompt_ids, tokenizer)

In [25]:
# Quick SFT sanity check without graph augmentation
baseline_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

sft_baseline_config = SFTConfig(
    output_dir='/Users/cyberlives/Documents/GitHub/GREP-PRISM/output',
    max_steps=100,
    max_seq_length=None,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=1,
    logging_steps=1,
    logging_strategy="steps",
    save_strategy="no",
    #evaluation_strategy="no",
    save_safetensors=False,
    report_to="none",
    dataloader_pin_memory=False,
)

train_subset = train_dataset.select(range(8))

'''
baseline_trainer = SFTTrainer(
    model=model,
    args=sft_baseline_config,
    train_dataset=train_subset,
    processing_class=tokenizer,
    data_collator=baseline_collator,
)

# baseline_train_result = baseline_trainer.train()
# baseline_train_result
'''

'\nbaseline_trainer = SFTTrainer(\n    model=model,\n    args=sft_baseline_config,\n    train_dataset=train_subset,\n    processing_class=tokenizer,\n    data_collator=baseline_collator,\n)\n\n# baseline_train_result = baseline_trainer.train()\n# baseline_train_result\n'

In [26]:
pe_model = RandomGNNPositionalEncodings(
    pe_hidden_channels=256, pe_num_layers=3, d_model=emb_dim, num_samples=40, dropout=0.1, # 0.05 0.01
    use_layer_norm=True
).to(device)

graph_augmented_model = GraphAugmentedLLM(model, pe_model, tokenizer).to(device)

collator = DataCollatorForGraphAugmentedLLM(tokenizer=tokenizer, mlm=False)

baseline_trainer = SFTTrainer(
    model=graph_augmented_model,
    args=sft_baseline_config,
    train_dataset=train_dataset,
    processing_class=tokenizer,
    data_collator=collator,
)

baseline_train_result = baseline_trainer.train()
baseline_train_result

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151645}.


Step,Training Loss
1,1.167800
2,1.022700
3,0.975100
4,0.922400
5,0.755800
6,0.636600
7,0.649600
8,0.443200
9,0.721800
10,0.720300


TrainOutput(global_step=100, training_loss=0.3766151374951005, metrics={'train_runtime': 111.7689, 'train_samples_per_second': 0.895, 'train_steps_per_second': 0.895, 'total_flos': 170300806021632.0, 'train_loss': 0.3766151374951005})